In [1]:
import os 
import warnings
from dotenv import load_dotenv


os.environ["KMP_DUPLICATE_LIB_OK"]="True"
warnings.filterwarnings("ignore")

load_dotenv()


True

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path="./rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf"
loader=PyMuPDFLoader(file_path)

docs=loader.load()

In [3]:
docs

[Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': './rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf', 'file_path': './rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-10-21T11:38:50+00:00', 'trapped': '', 'modDate': 'D:20241021113850Z', 'creationDate': '', 'page': 0}, page_content='Citation: Espeño, P.R.; Ong, A.K.S.;\nGerman, J.D.; Gumasing, M.J.J.; Casas,\nE.S. Analysis of Actual Fitness\nSupplement Consumption among\nHealth and Fitness Enthusiasts. Foods\n2024, 13, 1424. https://doi.org/\n10.3390/foods13091424\nAcademic Editors: Ilija Djekic\nand Nada Smigic\nReceived: 30 March 2024\nRevised: 15 April 2024\nAccepted: 18 April 2024\nPublished: 6 May 2024\nCopyright: © 2024 by the authors.\nLicensee MDPI, Basel, Switzerland.\nThis article is an open access article\ndistributed\nunde

In [4]:
doc=docs[0]

In [5]:
doc.metadata


{'producer': 'iLovePDF',
 'creator': '',
 'creationdate': '',
 'source': './rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf',
 'file_path': './rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf',
 'total_pages': 15,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2024-10-21T11:38:50+00:00',
 'trapped': '',
 'modDate': 'D:20241021113850Z',
 'creationDate': '',
 'page': 0}

In [6]:
print(doc.page_content)

Citation: Espeño, P.R.; Ong, A.K.S.;
German, J.D.; Gumasing, M.J.J.; Casas,
E.S. Analysis of Actual Fitness
Supplement Consumption among
Health and Fitness Enthusiasts. Foods
2024, 13, 1424. https://doi.org/
10.3390/foods13091424
Academic Editors: Ilija Djekic
and Nada Smigic
Received: 30 March 2024
Revised: 15 April 2024
Accepted: 18 April 2024
Published: 6 May 2024
Copyright: © 2024 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
distributed
under
the
terms
and
conditions of the Creative Commons
Attribution (CC BY) license (https://
creativecommons.org/licenses/by/
4.0/).
foods
Article
Analysis of Actual Fitness Supplement Consumption among
Health and Fitness Enthusiasts
Paolo Renzo Espeño 1, Ardvin Kester S. Ong 1,2,*
, Josephine D. German 1
, Ma. Janice J. Gumasing 3
and Ethan S. Casas 1
1
School of Industrial Engineering and Engineering Management, Mapúa University, 658 Muralla St.,
Intramuros, Manila 1002, Philippines
2
E.T. Yuchengo Scho

In [7]:
import os
pdfs=[]
for root, dirs, files in os.walk('rag-dataset'):
    # print(root,dirs,files)
    for file in files:
        if file.endswith('.pdf'):
            pdfs.append(os.path.join(root,file))
    
   

In [8]:
pdfs

['rag-dataset\\gym supplements\\1. Analysis of Actual Fitness Supplement.pdf',
 'rag-dataset\\gym supplements\\2. High Prevalence of Supplement Intake.pdf',
 'rag-dataset\\health supplements\\1. dietary supplements - for whom.pdf',
 'rag-dataset\\health supplements\\2. Nutraceuticals research.pdf',
 'rag-dataset\\health supplements\\3.health_supplements_side_effects.pdf']

In [9]:
docs=[]
for pdf in pdfs:
    loader=PyMuPDFLoader(pdf)
    pages=loader.load()
    docs.extend(pages)

In [10]:
len(docs)

64

### Document Chunking

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)

chunks=text_splitter.split_documents(docs)

In [12]:
# len(docs),len(chunks)
# print(docs[0].page_content)
# #see the difference between page content of chunks and docs
# print("page content of docs")
# print(chunks[0].page_content)

In [13]:
#to check the token
import tiktoken
encoding=tiktoken.encoding_for_model("gpt-4o-mini")

len(encoding.encode(docs[0].page_content))

968

## Document Vectorization

In [14]:
from langchain_ollama import OllamaEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore


In [15]:
embeddings=OllamaEmbeddings(model='nomic-embed-text',base_url="http://localhost:11434")

In [16]:
single_vector=embeddings.embed_query("this is some test text")
len(single_vector)

768

In [17]:
print(single_vector)

[0.02731011, 0.012167641, -0.16669317, -0.02726553, 0.05818836, -0.02560913, 0.030169658, -0.032787528, 0.009109163, -0.028767437, -0.01662998, 0.058169387, 0.023298979, 0.008039779, -0.056532692, -0.047375936, 0.056016564, -0.04969701, -0.024635026, 0.012108153, 0.0017161544, 0.02028042, -0.10023889, -0.028220275, 0.081333406, 0.03825225, -0.06937461, 0.03728661, -0.060225528, -0.038687576, 0.03616893, -0.048517775, 0.023452075, -0.045067534, -0.058201022, -0.03807013, 0.029144464, 0.046149846, -0.022927564, 0.024265863, 0.008950325, 0.0033069504, -0.021842407, -0.06339301, 0.043674197, -0.012516487, -0.011112689, 0.057557933, -0.008183639, -0.026551817, -0.016922653, -0.027704446, 0.017290324, -0.052824926, 0.04612792, 0.00085839326, -0.008236081, -0.040135212, 0.018261274, -0.03926346, 0.07362304, 0.072160095, -0.11688594, 0.04656298, 0.070141226, -0.067199945, -0.04285268, 0.014098444, -0.0026477915, 0.019427057, 0.07957743, 0.0298302, -0.004577195, -0.010733076, -0.053190928, -0.0

In [18]:
print(single_vector[:3])

[0.02731011, 0.012167641, -0.16669317]


In [19]:
index = faiss.IndexFlatL2(len(single_vector))

In [20]:
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [21]:
vector_store

In [22]:
# help(vector_store)

In [23]:
ids=vector_store.add_documents(documents=chunks)

In [24]:
#this will store the id of each chunk in index_to_docstore_id internally

In [25]:
ids

['a5c38113-f3c2-4147-805f-91dca5ed0daa',
 '50c93101-5e12-4f94-b9bb-ac80f31f9dd7',
 '1f524cd7-aadb-4f66-8ec3-1ec55867cd25',
 '02ab783a-de76-415f-9abf-d483e506c420',
 '6a84f6dc-a13c-4eb0-be79-b9cf20ff7e83',
 'd0defe5b-a3eb-409a-ae65-3698f6c30414',
 'bdabc2f7-1bd6-4d36-a3d3-b094e7d8629d',
 'd5d9c152-27b0-4ab5-9c1f-23371de93214',
 '18cb10e4-2150-4a6c-9aad-9e602aac22fc',
 '89f746ee-521a-4e77-90d1-d5efa56e8190',
 'a19576c8-8376-4f1f-ae63-9ecf4927f924',
 '297fae93-0696-46ac-8741-908c52155d2e',
 '6379e158-5217-4e8a-8048-11b033c9ae07',
 '874aebf8-f6b2-4514-be80-f8e91e3e7157',
 'a204f3d3-0471-40da-b8ed-32da11c8c003',
 'b722532b-fd2a-4ed9-b3f7-67864bb1acbe',
 'a659c67a-bbfb-43c8-b4b7-abb45c21f0dc',
 'c3e2f28b-5691-4b02-9679-7717259cc8d6',
 '43bff62d-0494-4e96-9be0-696f4dd437ef',
 'e4c484e8-0928-4a61-aa7f-a281d6e229a7',
 'ef40ad45-a49c-4400-b593-18c33b6ee4e7',
 '130bda55-2e4a-4d65-b5b0-333b7e8edead',
 'f28ff327-d3cd-4034-8de1-59b8f667fe59',
 '02554648-b9c6-4928-8635-5f6202d248b9',
 '98423dae-7b2b-

In [26]:
vector_store.index_to_docstore_id

{0: 'a5c38113-f3c2-4147-805f-91dca5ed0daa',
 1: '50c93101-5e12-4f94-b9bb-ac80f31f9dd7',
 2: '1f524cd7-aadb-4f66-8ec3-1ec55867cd25',
 3: '02ab783a-de76-415f-9abf-d483e506c420',
 4: '6a84f6dc-a13c-4eb0-be79-b9cf20ff7e83',
 5: 'd0defe5b-a3eb-409a-ae65-3698f6c30414',
 6: 'bdabc2f7-1bd6-4d36-a3d3-b094e7d8629d',
 7: 'd5d9c152-27b0-4ab5-9c1f-23371de93214',
 8: '18cb10e4-2150-4a6c-9aad-9e602aac22fc',
 9: '89f746ee-521a-4e77-90d1-d5efa56e8190',
 10: 'a19576c8-8376-4f1f-ae63-9ecf4927f924',
 11: '297fae93-0696-46ac-8741-908c52155d2e',
 12: '6379e158-5217-4e8a-8048-11b033c9ae07',
 13: '874aebf8-f6b2-4514-be80-f8e91e3e7157',
 14: 'a204f3d3-0471-40da-b8ed-32da11c8c003',
 15: 'b722532b-fd2a-4ed9-b3f7-67864bb1acbe',
 16: 'a659c67a-bbfb-43c8-b4b7-abb45c21f0dc',
 17: 'c3e2f28b-5691-4b02-9679-7717259cc8d6',
 18: '43bff62d-0494-4e96-9be0-696f4dd437ef',
 19: 'e4c484e8-0928-4a61-aa7f-a281d6e229a7',
 20: 'ef40ad45-a49c-4400-b593-18c33b6ee4e7',
 21: '130bda55-2e4a-4d65-b5b0-333b7e8edead',
 22: 'f28ff327-d3cd-

In [27]:
#to store the vector store locally
# db_name="health_supplements"
# vector_store.save_local(db_name)

In [28]:

#to load the db have to give db name and embeddings used to make the vector which was saved in the db 
#FAISS.load_local(db_name,embedding used,approval for deserialization)
# new_vector_store=FAISS.load_local("health_supplements",embeddings=OllamaEmbeddings(model='nomic-embed-text',base_url="http://localhost:11434"),allow_dangerous_deserialization=True)

## Retreival

In [29]:
# question="what is used to gain muscle mass ?"
# docs=vector_store.search(query=question,search_type="similarity")

# for doc in docs:
#     print(doc.page_content)
#     print("\n\n")
    

# this is how query is  done now 
#commenting all because everytime we run this code will rerun and may cause some issue


### making retiever for the vector store


In [30]:
retriever=vector_store.as_retriever(search_type="mmr",search_kwargs={'k':3,'fetch_k':100,'lambda_mult':1})

In [ ]:
# question="what is used to gain muscle mass ?"
# docs=retriever.invoke(question)
# for doc in docs:
#     print(doc.page_content)
#     print("\n\n")

acids than traditional protein sources. Its numerous benefits have made it a popular choice
for snacks and drinks among consumers [3]. Another widely embraced supplement is
caffeine, which is found in many sports and food supplements. Caffeine reduces perceived
effort, minimizes fatigue and pain, and proves to be effective for endurance and high-
intensity activities, which is the choice of consumers [4].
Creatine monohydrate is another well-known supplement used to gain muscle mass
and support performance and recovery. It is known not to increase fat mass and remains
effective even when taken in recommended doses [5]. Despite its popularity in the fitness
Foods 2024, 13, 1424. https://doi.org/10.3390/foods13091424
https://www.mdpi.com/journal/foods



and strength gain among men. We detected more prevalent protein and creatine supplementation
among younger compared to older ﬁtness center users, whereas the opposite was found for vitamin
supplementation. Other authors made similar obse

In [ ]:
# question="what are the benefit of BCAA supplements?"
# docs=retriever.invoke(question)

#asking this question down the code with prompt

In [ ]:
# question="what is used to reduce weight?"
# docs=retriever.invoke(question)
#asking this question down the code with prompt

## RAG with LLAMA 3.2 on OLLAMA

In [36]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser  #StrOutputParser is used to parse the output given by llama llm in string
from langchain_core.runnables import RunnablePassthrough  #Runnable Passthrough is used to pass the question directly tp llama llm and also passthrough passes the chunks to llama llm 
from langchain_core.prompts import ChatPromptTemplate

from langchain_ollama import ChatOllama

In [ ]:
model=ChatOllama(model="llama3.2:3b",base_url="http://localhost:11434")  #llama 3.2 3b model is used
model.invoke("hi")

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-08-16T12:58:07.6088651Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6301241900, 'load_duration': 5404601000, 'prompt_eval_count': 26, 'prompt_eval_duration': 631363300, 'eval_count': 8, 'eval_duration': 255068100, 'model_name': 'llama3.2:3b'}, id='run--ddb42c66-74e3-432c-8d5b-76f6309bce2c-0', usage_metadata={'input_tokens': 26, 'output_tokens': 8, 'total_tokens': 34})

In [40]:
prompt=hub.pull("rlm/rag-prompt")

In [41]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [48]:
#for prompt we can either take from hub by hub.pull(prompt package)  here prompt package is rlm/rag-prompt
#or else we can create our custom prompt and pass it through ChatPromptTemplate.from_template



prompt="""
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. 
If possible answer in bullet points. Make sure your answer is relevant to the question and it is answered from the context only.
Question:{question}
Context:{context}
Answer:

"""

prompt=ChatPromptTemplate.from_template(prompt)

In [ ]:
def format(docs):
    return "\n\n".join([doc.page_content for doc in docs])
# print(format(docs))
#now commented this

are used by as many as 40–50% of young women, regardless of their weight [15].
Weight loss supplements are usually multi-ingredient preparations, with over 4000 in-
dividual substances used in the production process. The average weight loss supplement
available in Western markets is estimated to include 10 different ingredients [109]. The
more complex the recipe, the harder it is to determine its effects on the body. The most
popular ingredients include chromium and chitosan, as well as green tea, Garcinia cambogia,
and bitter orange (Citrus aurantium) extracts [15,16]. Over the years, no studies have shown
that the use of either single- or multi-ingredient preparations of those substances promotes
weight reduction.
A 2013 meta-analysis of randomized studies found that chromium supplementation
resulted in only 0.5 kg additional weight reduction in subjects with overweight and obesity,
as compared with those taking a placebo [110], and a comparable result (mean: 0.75 kg)

Int. J. Enviro

In [50]:
rag_chain=(
    {
        "context":retriever|format,
        "question":RunnablePassthrough()
    }
    |prompt
    |model
    |StrOutputParser()
)

In [51]:
# question="what is used to gain muscle mass?"
question="what are the benefit of BCAA supplements?"
# question="what is used to reduce weight?"
# question="what are side effect of supplements?"
# question="what are benefits of supplements?"

output=rag_chain.invoke(question)
print(output)

Here are the benefits of BCAA supplements in bullet points, based on the context provided:

• Contribute to the process of strengthening muscles and alleviating post-workout soreness.
• Serve as raw materials needed to build new muscle, stored directly in muscles.

Note that the context does not provide a comprehensive list of benefits or specific advantages of BCAA supplements. The above two points are mentioned as examples of their role in supporting muscle growth and recovery.


In [52]:
# question="what is used to gain muscle mass?"
# question="what are the benefit of BCAA supplements?"
question="what is used to reduce weight?"
# question="what are side effect of supplements?"
# question="what are benefits of supplements?"

output=rag_chain.invoke(question)
print(output)

Here are the points about what is not used to reduce weight, based on the provided context:

• Over 4000 individual substances are used in the production of weight loss supplements, but none have been shown to promote weight reduction through single- or multi-ingredient preparations.

• Chromium supplementation has resulted in only a minimal amount of additional weight reduction (0.5 kg) compared to those taking a placebo.

• Studies have not found that the use of chromium, chitosan, green tea, Garcinia cambogia, and bitter orange extracts promotes weight loss.

• A 2013 meta-analysis found that supplementation with these substances did not result in significant weight reduction.

Note: The text does mention that some people may still try to lose weight using dietary supplements, despite the lack of evidence for their effectiveness. However, it also emphasizes the importance of being cautious and not purchasing products from unauthorized sources or increasing the recommended dose.
